In [6]:
!pip install -q transformers sentence-transformers fuzzywuzzy[speedup] xgboost lightgbm

import pandas as pd
import numpy as np
import re, html
from tqdm import tqdm

import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [2]:
from google.colab import files

uploaded = files.upload()

Saving questions.csv to questions.csv


In [7]:
df = pd.read_csv('questions.csv')  # adjust path/columns
df = df.dropna(subset=['question1','question2','is_duplicate'])
df.head()


,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,1,3,4,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,2,5,6,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,3,7,8,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,4,9,10,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0


In [9]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')   # <-- add this line
nltk.download('stopwords')
nltk.download('wordnet')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [10]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

CONTRACTIONS = {
    "can't":"can not","won't":"will not","don't":"do not","i'm":"i am",
    "it's":"it is","you're":"you are","they're":"they are","that's":"that is"
    # extend as needed
}

def decontract(text):
    text = text.lower()
    for c, full in CONTRACTIONS.items():
        text = re.sub(r"\b" + re.escape(c) + r"\b", full, text)
    return text

def clean_text(text):
    text = html.unescape(text)
    text = re.sub(r'<.*?>', ' ', text)        # remove HTML tags
    text = decontract(text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)  # keep alnum + space
    tokens = nltk.word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens
              if t not in stop_words and len(t) > 1]
    return ' '.join(tokens)

for col in ['question1','question2']:
    df[col + '_clean'] = df[col].astype(str).apply(clean_text)

df[['question1_clean','question2_clean']].head()


,question1_clean,question2_clean
0,step step guide invest share market india,step step guide invest share market
1,story kohinoor koh noor diamond,would happen indian government stole kohinoor ...
2,increase speed internet connection using vpn,internet speed increased hacking dns
3,mentally lonely solve,find remainder math 23 24 math divided 24 23
4,one dissolve water quikly sugar salt methane c...,fish would survive salt water


In [11]:
def jaccard(a, b):
    sa, sb = set(a.split()), set(b.split())
    if not sa and not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)

def longest_common_substring(a, b):
    # dynamic programming LCS (substring)
    m, n = len(a), len(b)
    dp = [[0]*(n+1) for _ in range(m+1)]
    longest = 0
    for i in range(m):
        for j in range(n):
            if a[i] == b[j]:
                dp[i+1][j+1] = dp[i][j] + 1
                longest = max(longest, dp[i+1][j+1])
    return longest

def length_features(q1, q2):
    l1, l2 = len(q1), len(q2)
    abs_diff = abs(l1 - l2)
    mean_len = (l1 + l2) / 2
    lcs = longest_common_substring(q1, q2)
    lcs_ratio = lcs / (min(l1, l2) + 1e-6)
    return abs_diff, mean_len, lcs_ratio

from fuzzywuzzy import fuzz

def build_basic_features(df):
    feats = {}
    feats['jaccard'] = df.apply(lambda r: jaccard(r['question1_clean'],
                                                  r['question2_clean']), axis=1)
    feats['len_diff'], feats['mean_len'], feats['lcs_ratio'] = zip(*df.apply(
        lambda r: length_features(r['question1_clean'], r['question2_clean']), axis=1))
    feats['fuzz_ratio'] = df.apply(lambda r: fuzz.QRatio(r['question1_clean'],
                                                         r['question2_clean']), axis=1)
    feats['fuzz_partial'] = df.apply(lambda r: fuzz.partial_ratio(r['question1_clean'],
                                                                  r['question2_clean']), axis=1)
    feats['fuzz_token_sort'] = df.apply(lambda r: fuzz.token_sort_ratio(r['question1_clean'],
                                                                        r['question2_clean']), axis=1)
    feats['fuzz_token_set'] = df.apply(lambda r: fuzz.token_set_ratio(r['question1_clean'],
                                                                      r['question2_clean']), axis=1)
    return pd.DataFrame(feats)

basic_feats = build_basic_features(df)
basic_feats.head()


,jaccard,len_diff,mean_len,lcs_ratio,fuzz_ratio,fuzz_partial,fuzz_token_sort,fuzz_token_set
0,0.833333,6,38.0,1.000000,92,100,92,100
1,0.363636,36,49.0,0.838710,59,94,59,89
2,0.222222,8,40.0,0.250000,55,56,70,70
3,0.000000,23,32.5,0.047619,22,24,22,22
4,0.153846,31,44.5,0.206897,43,52,40,51


In [12]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer('all-MiniLM-L6-v2')  # light but good

def bert_cosine_batch(q1_list, q2_list, batch_size=256):
    sims = []
    for i in range(0, len(q1_list), batch_size):
        q1_batch = q1_list[i:i+batch_size].tolist()
        q2_batch = q2_list[i:i+batch_size].tolist()
        e1 = model.encode(q1_batch, convert_to_numpy=True, show_progress_bar=False)
        e2 = model.encode(q2_batch, convert_to_numpy=True, show_progress_bar=False)
        s = np.diag(cosine_similarity(e1, e2))
        sims.extend(s)
    return np.array(sims)

df['bert_cosine'] = bert_cosine_batch(df['question1_clean'], df['question2_clean'])
df['bert_cosine'].head()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

,bert_cosine
0,0.915797
1,0.631126
2,0.422067
3,0.236699
4,0.225019


In [13]:
X = pd.concat([basic_feats, df[['bert_cosine']]], axis=1)
y = df['is_duplicate'].astype(int)


In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import xgboost as xgb
import lightgbm as lgb


In [15]:
def eval_model(clf, name):
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    acc = accuracy_score(y_test, preds)
    p, r, f1, _ = precision_recall_fscore_support(y_test, preds, average='binary')
    print(f'{name}: acc={acc:.3f}, precision={p:.3f}, recall={r:.3f}, f1={f1:.3f}')


In [16]:
lr = LogisticRegression(max_iter=200, n_jobs=-1)
rf = RandomForestClassifier(n_estimators=300, max_depth=None, n_jobs=-1, random_state=42)
gb = GradientBoostingClassifier(random_state=42)
xgb_clf = xgb.XGBClassifier(
    n_estimators=400, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8, objective='binary:logistic',
    tree_method='hist', eval_metric='logloss'
)
lgb_clf = lgb.LGBMClassifier(
    n_estimators=400, learning_rate=0.05, num_leaves=63,
    subsample=0.8, colsample_bytree=0.8, objective='binary'
)

for clf, name in [
    (lr, 'LogisticRegression'),
    (rf, 'RandomForest'),
    (gb, 'GradientBoosting'),
    (xgb_clf, 'XGBoost'),
    (lgb_clf, 'LightGBM')
]:
    eval_model(clf, name)


LogisticRegression: acc=0.745, precision=0.651, recall=0.666, f1=0.659
RandomForest: acc=0.782, precision=0.698, recall=0.724, f1=0.711
GradientBoosting: acc=0.765, precision=0.671, recall=0.715, f1=0.692
XGBoost: acc=0.775, precision=0.684, recall=0.726, f1=0.704
[LightGBM] [Info] Number of positive: 119445, number of negative: 204033
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.043782 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1489
[LightGBM] [Info] Number of data points in the train set: 323478, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.369252 -> initscore=-0.535426
[LightGBM] [Info] Start training from score -0.535426
LightGBM: acc=0.776, precision=0.684, recall=0.729, f1=0.706
